In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND EXPECTED CONTROLS
# ===================================================

from pyspark.sql import functions as F


"""
Validate completeness, file coverage, lineage, rescued data, and rerun
behavior for all remaining Bronze batch sources.
"""

expected_tables = {
    "semiconplus_portfolio.bronze.equipment_events": (255_640, 60),
    "semiconplus_portfolio.bronze.unit_test_results": (181_250, 60),
    "semiconplus_portfolio.bronze.tester_logs_raw": (18_125, 60),
    "semiconplus_portfolio.bronze.ref_devices": (30, 1),
    "semiconplus_portfolio.bronze.ref_equipment": (24, 1),
    "semiconplus_portfolio.bronze.ref_product_groups": (6, 1),
    "semiconplus_portfolio.bronze.ref_sites": (3, 1),
    "semiconplus_portfolio.bronze.maintenance_documents_binary": (25, 25),
    "semiconplus_portfolio.bronze.source_manifests_raw": (2, 2),
}

In [0]:
# ===================================================
# BLOCK 2 — TABLE AVAILABILITY AND ROW COUNTS
# ===================================================

"""
Confirm that every required Bronze table exists and matches its approved
initial-load record count.
"""

validation_results = []

for table_name, (expected_rows, expected_files) in expected_tables.items():
    assert spark.catalog.tableExists(table_name), (
        f"Required Bronze table does not exist: {table_name}"
    )

    actual_rows = spark.table(table_name).count()

    assert actual_rows == expected_rows, (
        f"{table_name}: expected {expected_rows:,} rows, "
        f"found {actual_rows:,}."
    )

    validation_results.append(
        (table_name, expected_rows, actual_rows, "PASSED")
    )

display(
    spark.createDataFrame(
        validation_results,
        ["table_name", "expected_rows", "actual_rows", "status"],
    )
)

In [0]:
# ===================================================
# BLOCK 3 — INCREMENTAL SOURCE FILE COVERAGE
# ===================================================

"""
Confirm that all 60 files from each incrementally processed source are
represented in the corresponding Bronze table.
"""

incremental_tables = [
    "semiconplus_portfolio.bronze.equipment_events",
    "semiconplus_portfolio.bronze.unit_test_results",
    "semiconplus_portfolio.bronze.tester_logs_raw",
]

for table_name in incremental_tables:
    table_df = spark.table(table_name)
    file_count = table_df.select("_source_file_path").distinct().count()

    print(f"{table_name}: source files={file_count}")
    assert file_count == 60

print("Incremental source-file coverage passed.")

In [0]:
# ===================================================
# BLOCK 4 — LINEAGE COMPLETENESS
# ===================================================

"""
Confirm that all incrementally ingested records contain the operational
metadata required for file and pipeline-run traceability.
"""

lineage_columns = [
    "_source_file_path",
    "_source_file_name",
    "_source_file_modification_time",
    "_ingested_at_utc",
    "_pipeline_run_id",
]

for table_name in incremental_tables:
    table_df = spark.table(table_name)

    lineage_failures = table_df.filter(
        F.col("_source_file_path").isNull()
        | F.col("_source_file_name").isNull()
        | F.col("_source_file_modification_time").isNull()
        | F.col("_ingested_at_utc").isNull()
        | F.col("_pipeline_run_id").isNull()
    ).count()

    print(f"{table_name}: lineage failures={lineage_failures}")
    assert lineage_failures == 0

print("Incremental Bronze lineage validation passed.")

In [0]:
# ===================================================
# BLOCK 5 — RESCUED-DATA REVIEW
# ===================================================

"""
Report source records containing fields that did not match the tracked
Auto Loader schema.

Any non-zero result must be reviewed before building the corresponding
Silver transformation.
"""

rescued_results = []

for table_name in incremental_tables:
    table_df = spark.table(table_name)

    rescued_count = table_df.filter(
        F.col("_rescued_data").isNotNull()
    ).count()

    rescued_results.append((table_name, rescued_count))

display(
    spark.createDataFrame(
        rescued_results,
        ["table_name", "rescued_record_count"],
    )
)

assert all(count == 0 for _, count in rescued_results), (
    "Rescued data was detected. Review affected records before Silver."
)

print("Rescued-data validation passed.")

In [0]:
# ===================================================
# BLOCK 6 — REFERENCE BUSINESS-KEY VALIDATION
# ===================================================


"""
Confirm that reference datasets contain unique, non-null business keys.
"""

reference_keys = {
    "semiconplus_portfolio.bronze.ref_devices": "device_id",
    "semiconplus_portfolio.bronze.ref_equipment": "equipment_id",
    "semiconplus_portfolio.bronze.ref_product_groups": "product_group_id",
    "semiconplus_portfolio.bronze.ref_sites": "site_id",
}

for table_name, key_column in reference_keys.items():
    table_df = spark.table(table_name)

    invalid_key_count = table_df.filter(
        F.col(key_column).isNull()
        | (F.trim(F.col(key_column)) == "")
    ).count()

    duplicate_key_count = (
        table_df
        .groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{table_name}: invalid keys={invalid_key_count}, "
        f"duplicate keys={duplicate_key_count}"
    )

    assert invalid_key_count == 0
    assert duplicate_key_count == 0

print("Reference business-key validation passed.")

In [0]:
# ===================================================
# BLOCK 7 — BINARY AND MANIFEST VALIDATION
# ===================================================

"""
Confirm that binary document payloads and decoded source manifests are
present and retain unique source paths.
"""

binary_df = spark.table(
    "semiconplus_portfolio.bronze.maintenance_documents_binary"
)

manifest_df = spark.table(
    "semiconplus_portfolio.bronze.source_manifests_raw"
)

assert binary_df.filter(
    F.col("binary_content").isNull()
    | (F.col("content_length_bytes") <= 0)
).count() == 0

assert binary_df.select("source_file_path").distinct().count() == 25

assert manifest_df.filter(
    F.col("json_payload").isNull()
    | (F.length(F.col("json_payload")) == 0)
).count() == 0

assert manifest_df.select("source_file_path").distinct().count() == 2

print("Binary-document and manifest validation passed.")

In [0]:
# ===================================================
# BLOCK 8 — INGESTION AUDIT VALIDATION
# ===================================================

"""
Confirm that the current ingestion execution produced successful audit
records for every incrementally processed source.
"""

AUDIT_TABLE = "semiconplus_portfolio.monitoring.ingestion_audit"

assert spark.catalog.tableExists(AUDIT_TABLE)

latest_audit_df = (
    spark.table(AUDIT_TABLE)
    .filter(F.col("run_status") == "SUCCEEDED")
    .orderBy(F.col("completed_at_utc").desc())
)

display(latest_audit_df.limit(10))

for source_name in [
    "equipment_events",
    "unit_test_results",
    "tester_logs",
]:
    assert latest_audit_df.filter(
        F.col("source_name") == source_name
    ).count() >= 1

print("Ingestion-audit validation passed.")

In [0]:
# ===================================================
# BLOCK 9 — CAPTURE COUNTS BEFORE RERUN
# ===================================================

"""
Capture current Auto Loader table counts before repeating the ingestion
with the existing checkpoints.
"""

counts_before_rerun = {
    table_name: spark.table(table_name).count()
    for table_name in incremental_tables
}

print(counts_before_rerun)


In [0]:
# ---------------
# IDEMPOTENCY RERUN PROCEDURE
# ---------------

# 1. Run Block 9.
# 2. Return to 05_bronze_remaining_sources.
# 3. Rerun Blocks 1 through 7 only.
# 4. Do not rerun the audit block for this test.
# 5. Return here without clearing the Python session.
# 6. Run Block 10.

# ===================================================
# BLOCK 10 — IDEMPOTENCY VALIDATION
# ===================================================

"""
Confirm that existing Auto Loader checkpoints prevent previously
processed files from being appended during a repeated ingestion run.
"""

counts_after_rerun = {
    table_name: spark.table(table_name).count()
    for table_name in incremental_tables
}

for table_name in incremental_tables:
    before_count = counts_before_rerun[table_name]
    after_count = counts_after_rerun[table_name]
    rows_added = after_count - before_count

    print(
        f"{table_name}: before={before_count:,}, "
        f"after={after_count:,}, added={rows_added:,}"
    )

    assert rows_added == 0

print("REMAINING BRONZE IDEMPOTENCY TEST PASSED")

In [0]:
# ===================================================
# BLOCK 11 — FINAL VALIDATION RESULT
# ===================================================

"""
Publish the final remaining-source Bronze validation result for project
documentation and execution evidence.
"""

print("REMAINING BRONZE VALIDATION PASSED")
print("Equipment-event rows: 255,640")
print("Unit-test-result rows: 181,250")
print("Tester-log rows: 18,125")
print("Reference entities: 63")
print("Binary documents: 25")
print("Source manifests: 2")
print("Streaming-event files intentionally pending: 60")